In [25]:
import pandas as pd

df= pd.read_csv(r'C:\Users\daniealv\Downloads\prueba_op_base_pivot_var_rpta_alt_enmascarado_trtest.csv')
df3= pd.read_csv(r'C:\Users\daniealv\Downloads\prueba_op_probabilidad_oblig_base_hist_enmascarado_completa.csv')
df3 = df3.drop_duplicates()
df3['prob_alrt_temprana'] = pd.to_numeric(df3['prob_alrt_temprana'], errors='coerce')
df3['prob_auto_cura'] = pd.to_numeric(df3['prob_auto_cura'], errors='coerce')
extracted_df = df[['nit_enmascarado', 'num_oblig_enmascarado', 'num_oblig_orig_enmascarado','var_rpta_alt','fecha_var_rpta_alt']]


c:\Users\daniealv\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3444: DtypeWarning: Columns (28,37,38) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)
c:\Users\daniealv\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3444: DtypeWarning: Columns (5,6) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [41]:
tipos=df3.dtypes
describe=df3.describe()

In [39]:
missing_values = df3.isnull().sum()
print(missing_values)

nit_enmascarado          0
num_oblig_enmascarado    0
fecha_corte              0
lote                     0
prob_propension          0
prob_alrt_temprana       7
prob_auto_cura           7
dtype: int64


In [ ]:
missing_values = merged_check.isnull().sum()
print(missing_values)

nit_enmascarado                   0
num_oblig_enmascarado             0
num_oblig_orig_enmascarado        0
var_rpta_alt                      0
fecha_var_rpta_alt                0
prob_auto_cura_1               1816
prob_auto_cura_2               5597
prob_auto_cura_3              18029
prob_auto_cura_4              40789
prob_auto_cura_5              63149
prob_auto_cura_6              82201
prob_propension_1              1811
prob_propension_2              5595
prob_propension_3             18029
prob_propension_4             40789
prob_propension_5             63149
prob_propension_6             82201
prob_alrt_temprana_1           1816
prob_alrt_temprana_2           5597
prob_alrt_temprana_3          18029
prob_alrt_temprana_4          40789
prob_alrt_temprana_5          63149
prob_alrt_temprana_6          82201
dtype: int64


In [32]:
merged_df = pd.merge(extracted_df, df3, how='left', on=['nit_enmascarado', 'num_oblig_enmascarado'])
# Convertir las columnas 'fecha_corte' y 'fecha_var_rpta_alt' en tipo datetime
merged_df['fecha_corte'] = pd.to_datetime(merged_df['fecha_corte'], format='%Y%m')
merged_df['fecha_var_rpta_alt'] = pd.to_datetime(merged_df['fecha_var_rpta_alt'], format='%Y%m')
merged_df = merged_df[merged_df['fecha_var_rpta_alt'] > merged_df['fecha_corte']]
extracted_df['fecha_var_rpta_alt'] = pd.to_datetime(extracted_df['fecha_var_rpta_alt'], format='%Y%m')


C:\Users\daniealv\AppData\Local\Temp/ipykernel_29084/2459027368.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  extracted_df['fecha_var_rpta_alt'] = pd.to_datetime(extracted_df['fecha_var_rpta_alt'], format='%Y%m')


In [33]:
# Ordenar el DataFrame por 'num_oblig_enmascarado', 'fecha_var_rpta_alt' y 'fecha_corte'
merged_df = merged_df.sort_values(by=['num_oblig_enmascarado','num_oblig_orig_enmascarado', 'fecha_var_rpta_alt', 'fecha_corte'])

# Tomar los últimos seis pagos para cada obligación
merged_df = merged_df.groupby(['num_oblig_enmascarado','num_oblig_orig_enmascarado','fecha_var_rpta_alt']).tail(6)
merged_df['numerador'] = merged_df.groupby(['nit_enmascarado', 'num_oblig_enmascarado','num_oblig_orig_enmascarado','fecha_var_rpta_alt']).cumcount() + 1
pivot_df = merged_df.pivot_table(index=['nit_enmascarado', 'num_oblig_enmascarado','num_oblig_orig_enmascarado','fecha_var_rpta_alt'], columns='numerador', values='prob_auto_cura').reset_index()
# Renombrar las columnas para que sean más descriptivas
pivot_df.columns = ['nit_enmascarado', 'num_oblig_enmascarado','num_oblig_orig_enmascarado','fecha_var_rpta_alt'] + [f'prob_auto_cura_{i}' for i in range(1, 7)]
# Realizar un merge para encontrar los registros en extracted_df que no están en pivot_df
merged_check = pd.merge(extracted_df, pivot_df, how='left', on=['nit_enmascarado', 'num_oblig_enmascarado', 'num_oblig_orig_enmascarado','fecha_var_rpta_alt'])

In [34]:
pivot_df = merged_df.pivot_table(index=['nit_enmascarado', 'num_oblig_enmascarado','num_oblig_orig_enmascarado','fecha_var_rpta_alt'], columns='numerador', values='prob_propension').reset_index()
# Renombrar las columnas para que sean más descriptivas
pivot_df.columns = ['nit_enmascarado', 'num_oblig_enmascarado','num_oblig_orig_enmascarado','fecha_var_rpta_alt'] + [f'prob_propension_{i}' for i in range(1, 7)]
# Realizar un merge para encontrar los registros en extracted_df que no están en pivot_df
merged_check = pd.merge(merged_check, pivot_df, how='left', on=['nit_enmascarado', 'num_oblig_enmascarado', 'num_oblig_orig_enmascarado','fecha_var_rpta_alt'])

In [35]:
pivot_df = merged_df.pivot_table(index=['nit_enmascarado', 'num_oblig_enmascarado','num_oblig_orig_enmascarado','fecha_var_rpta_alt'], columns='numerador', values='prob_alrt_temprana').reset_index()
# Renombrar las columnas para que sean más descriptivas
pivot_df.columns = ['nit_enmascarado', 'num_oblig_enmascarado','num_oblig_orig_enmascarado','fecha_var_rpta_alt'] + [f'prob_alrt_temprana_{i}' for i in range(1, 7)]
# Realizar un merge para encontrar los registros en extracted_df que no están en pivot_df
merged_check = pd.merge(merged_check, pivot_df, how='left', on=['nit_enmascarado', 'num_oblig_enmascarado', 'num_oblig_orig_enmascarado','fecha_var_rpta_alt'])

In [36]:
# Calcular la correlación entre 'var_rpta_alt' y las demás variables
correlation_matrix = merged_check.corr()
var_rpta_correlation = correlation_matrix['var_rpta_alt'].sort_values(ascending=False)
print(var_rpta_correlation)

var_rpta_alt                  1.000000
prob_propension_5             0.189754
prob_auto_cura_5              0.188926
prob_auto_cura_4              0.187775
prob_auto_cura_3              0.187186
prob_propension_4             0.182742
prob_auto_cura_2              0.177413
prob_propension_3             0.176234
prob_auto_cura_1              0.170468
prob_propension_2             0.156741
prob_auto_cura_6              0.154451
prob_propension_6             0.152409
prob_propension_1             0.142784
num_oblig_enmascarado         0.049799
nit_enmascarado               0.019771
num_oblig_orig_enmascarado   -0.044118
prob_alrt_temprana_6         -0.131197
prob_alrt_temprana_1         -0.131829
prob_alrt_temprana_2         -0.151947
prob_alrt_temprana_5         -0.163647
prob_alrt_temprana_4         -0.167509
prob_alrt_temprana_3         -0.168739
Name: var_rpta_alt, dtype: float64


In [11]:
# Imputar valores faltantes con el promedio
final_df['prob_propension'] = final_df['prob_propension'].fillna(final_df['prob_propension'].mean())
final_df['prob_alrt_temprana'] = final_df['prob_alrt_temprana'].fillna(final_df['prob_alrt_temprana'].mean())
final_df['prob_auto_cura'] = final_df['prob_auto_cura'].fillna(final_df['prob_auto_cura'].mean())